<div align="center">

# Week 10: Accessibility and Inclusive Design

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-10) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/Week10_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the age groups of the clinic's patients with their limitations and the confirmation texts in English and Turkish. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import re
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The patients of the clinic by age group (Week 9) and the share of each age
# group that cannot meet a demand of the current kiosk (lecture, Section 1)
AGE_MIX = {'under 40': 0.25, '40 to 64': 0.4, '65 and over': 0.35}
LIMITS = {'reading small text': [0.04, 0.08, 0.22],
          'reaching high buttons': [0.02, 0.05, 0.19],
          'acting within 30 s': [0.01, 0.03, 0.14]}

# The confirmation of the current app and its plain version, in English and
# Turkish (lecture, Section 4)
TEXTS = {
    "current": (
        'Your appointment request has been successfully registered in the '
        'outpatient scheduling system. Please present your identification document '
        'at the registration desk fifteen minutes before your scheduled '
        'appointment time. Cancellations must be communicated at least twenty-four '
        'hours in advance.'),
    "plain": (
        'Your appointment is booked. Please come to the front desk fifteen minutes '
        'early and bring your health card. If you cannot come, please cancel one '
        'day before.'),
    "current_tr": (
        'Randevu talebiniz poliklinik randevu sistemine başarıyla kaydedilmiş olup '
        'randevu saatinizden on beş dakika önce kimlik belgenizle birlikte kayıt '
        'bankosuna müracaat etmeniz gerekmektedir. İptal taleplerinin en az yirmi '
        'dört saat önceden bildirilmesi gerekmektedir.'),
    "plain_tr": (
        'Randevunuz alındı. Lütfen on beş dakika erken gelin ve sağlık kartınızı '
        'getirin. Gelemezseniz bir gün önce iptal edin.')}
print("Setup complete. Running in Colab:", IN_COLAB, "| age groups:",
      list(AGE_MIX), "| texts:", list(TEXTS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 10.1": (lambda: len(re.findall(r'[A-Za-z]+', 'Bring your health card to the front desk.')), 1e-9),
    "Exercise 10.2": (lambda: len(re.findall(r'[aeıioöuü]', 'kartınızı')), 1e-9),
    "Exercise 10.3": (lambda: len(re.findall(r'\d{2}:\d{2}', 'Open 08:30 to 12:30 and 13:30 to 17:00.')), 1e-9),
    "Exercise A": (lambda: 1 - 0.88 * 0.91 * 0.94, 1e-6),
    "Exercise B": (lambda: 0.99 ** 55, 1e-6),
    "Exercise C": (lambda: stats.norm.cdf((1.62 - 1.55) / 0.10), 1e-6),
    "Exercise D": (lambda: fkgl(TEXTS['plain']), 1e-6),
    "Exercise E": (lambda: (5 + 1) / 2 + (30 / 5 + 1) / 2, 1e-9),
    "Exercise F": (lambda: coverage(1.13, True), 1e-6),
}

## Python step 10: Text with regular expressions

Accessibility is often a question of text: The words that a screen reader speaks and the sentences that patients read. This step handles text in Python &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-1)&#93;. It uses the methods of strings and the module `re` for regular expressions, and it counts words, syllables and sentences, the inputs of the reading formulas of Section 4.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "TEXTS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Strings and their methods

In [ ]:
# Python practice: a small example for learning the language
note = "Please come to the front desk fifteen minutes early."
print(note.lower())
print(note.split())                 # a list of the words, split at spaces
print("number of pieces:", len(note.split()))
print(note.replace("fifteen", "15"))

A string is a piece of text. `lower` returns it in small letters, `split` cuts it at the spaces into a list, and `replace` swaps one piece for another. The string itself does not change: Each method returns a new string or list. The last piece of `split` still carries the full stop.

### Finding words and times with re

In [ ]:
# Python practice: a small example for learning the language
note = "Your appointment: Wednesday, 10:30. Please come at 10:15."
print(re.findall(r"[A-Za-z]+", note))       # words: runs of letters
print(re.findall(r"\d{2}:\d{2}", note))     # times: 2 digits, colon, 2 digits
print(re.sub(r"\d{2}:\d{2}", "TIME", note))

The module `re` searches text with *regular expressions*, short patterns for pieces of text. `[A-Za-z]+` means one or more letters, `\d` a digit and `{2}` exactly two of them. `findall` returns every match, and `sub` replaces each match. The `r` before the quotes keeps the backslashes as they are.

### Counting syllables

In [ ]:
# Python practice: a small example for learning the language
def vowel_runs(word):
    """Runs of vowels in an English word, the start of syllable counting."""
    return re.findall(r"[aeiouy]+", word.lower())

for word in ["appointment", "early", "card", "identification"]:
    print(word, vowel_runs(word), len(vowel_runs(word)))
word_tr = "randevunuz"                       # Turkish for "your appointment"
print(word_tr, len(re.findall(r"[aeıioöuü]", word_tr)), "syllables")

In English, a syllable usually holds one run of vowels, so counting the runs approximates the syllables: The word *ap-point-ment* has three. Section 4 adds rules for a silent *e*. In Turkish, every syllable holds exactly one vowel, so counting the vowels gives the syllables: The word *ran-de-vu-nuz* has four.

### Splitting a text into sentences

In [ ]:
# Python practice: a small example for learning the language
text = "Your appointment is booked. Please come early! Do you need help?"
sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
print(sentences)
print(len(sentences), "sentences,", len(re.findall(r"[A-Za-z]+", text)),
      "words")

`re.split` cuts the text at every full stop, exclamation mark or question mark. The piece after the last mark is empty, and the list comprehension keeps only the pieces that contain text. `s.strip()` removes the spaces at both ends, and an empty string counts as false.

**Quick check.** The function `vowel_runs` finds two runs in the word *time*. Why does the word have only one syllable, and which rule would fix the count?

<details><summary>Answer</summary>

The final *e* is silent. Section 4 subtracts one run when a word ends in *e*, except for endings such as *-le* in *table*.

</details>

### Exercises for Python step 10

**Exercise 10.1.** Count the words of the sentence *Bring your health card to the front desk.* with `re.findall` and store the number in `n_words`.

In [ ]:
n_words = None   # your answer

check("Exercise 10.1", n_words)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_words = len(re.findall(r"[A-Za-z]+", "Bring your health card to the front desk."))

check("Exercise 10.1", n_words)

**Exercise 10.2.** Count the syllables of the Turkish word *kartınızı* (your card) by counting its vowels, and store the number in `n_syl`.

In [ ]:
n_syl = None   # your answer

check("Exercise 10.2", n_syl)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_syl = len(re.findall(r"[aeıioöuü]", "kartınızı"))

check("Exercise 10.2", n_syl)

**Exercise 10.3.** Store in `n_times` how many times of the form 08:30 the text *Open 08:30 to 12:30 and 13:30 to 17:00.* contains.

In [ ]:
n_times = None   # your answer

check("Exercise 10.3", n_times)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_times = len(re.findall(r"\d{2}:\d{2}",
                         "Open 08:30 to 12:30 and 13:30 to 17:00."))

check("Exercise 10.3", n_times)

## 1. Who is excluded: Overlapping limitations &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-4)&#93;

*Lecture: [Who is excluded: Disability as a mismatch](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#who-is-excluded)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "sentences" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A design that demands several abilities excludes everyone who lacks one of them. Python computes the excluded share with the formula for independent limitations and with its bounds. A simulation then lets the limitations come together, as they do with age.

**From the formulas to the code.** The excluded share for independent limitations, $E = 1 - \prod_j (1 - e_j)$, is `exclusion_independent(shares)`, and its bounds $\max_j e_j \le E \le \min(1, \sum_j e_j)$ are `exclusion_bounds(shares)`. `simulate_patients(rho)` draws patients whose limitations are linked by the correlation $\rho$.

In [ ]:
def exclusion_independent(shares):
    """Share excluded by a design whose demands fail independently."""
    return 1 - float(np.prod([1 - e for e in shares]))

def exclusion_bounds(shares):
    """Smallest and largest excluded share, whatever the overlap."""
    return max(shares), min(1.0, sum(shares))

shares = [sum(w * p for w, p in zip(AGE_MIX.values(), LIMITS[k]))
          for k in LIMITS]
for k, e in zip(LIMITS, shares):
    print(f"{k}: {e:.4f} of all patients")
example = [0.12, 0.09, 0.06]
print("worked example, independent:",
      round(exclusion_independent(example), 3))
print("worked example, bounds:", exclusion_bounds(example))

**Python note.** `zip` walks through two lists side by side, here the shares of the age groups and the limitation of each group. Their products, added up, give the share of all patients, as in the weighting of Week 9. The worked example rounds the three shares to 0.12, 0.09 and 0.06.

**Exercise A.** The kiosk demands three abilities, which exclude 0.12, 0.09 and 0.06 of the patients. Compute the excluded share for independent limitations, first by hand and then with `exclusion_independent`, and store it in `excluded`.

In [ ]:
excluded = None   # your answer

check("Exercise A", excluded)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
excluded = exclusion_independent([0.12, 0.09, 0.06])

check("Exercise A", excluded)

In [ ]:
def simulate_patients(rho, n=100_000, seed=RANDOM_STATE):
    """Patients with an age group and three limitations linked by rho."""
    rng = np.random.default_rng(seed)
    groups = rng.choice(len(AGE_MIX), size=n, p=list(AGE_MIX.values()))
    cov = np.full((3, 3), rho) + np.eye(3) * (1 - rho)
    z = rng.multivariate_normal(np.zeros(3), cov, size=n)
    limit = np.array([[stats.norm.ppf(LIMITS[k][g]) for k in LIMITS]
                      for g in range(len(AGE_MIX))])
    has = z < limit[groups]        # True where a patient has a limitation
    return groups, has

for rho in (0.0, 0.5, 0.8):
    groups, has = simulate_patients(rho)
    excluded_sim = has.any(axis=1)
    by_age = [excluded_sim[groups == g].mean() for g in range(3)]
    print(f"rho = {rho}: excluded {excluded_sim.mean():.3f} | by age: "
          + ", ".join(f"{a} {b:.3f}" for a, b in zip(AGE_MIX, by_age)))
print("independent:", round(exclusion_independent(shares), 3),
      "| bounds:", tuple(round(b, 3) for b in exclusion_bounds(shares)))

**Python note.** Each patient gets three numbers from a normal distribution with the correlation $\rho$. A limitation is present when its number falls below the quantile of the patient's age group, so each group keeps its shares. A high $\rho$ makes the limitations come together. `has.any(axis=1)` asks, row by row, whether a patient has at least one limitation.

In [ ]:
# Figure: limitations by age group and the excluded share against the overlap
rhos = np.linspace(0, 0.9, 10)
exc = [simulate_patients(r)[1].any(axis=1).mean() for r in rhos]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
x = np.arange(len(AGE_MIX))
for i, (k, ps) in enumerate(LIMITS.items()):
    ax[0].bar(x + (i - 1) * 0.26, ps, width=0.26, label=k)
ax[0].set_xticks(x, list(AGE_MIX))
ax[0].set_ylabel("share of the age group")
ax[0].legend(fontsize=8)
low, high = exclusion_bounds(shares)
ax[1].fill_between(rhos, low, high, color="#E8E4F7", label="bounds")
ax[1].plot(rhos, exc, "o-", color="#5A4BB5", label="simulation")
ax[1].axhline(exclusion_independent(shares), ls="--", color="#B53F2E",
              label="independent")
ax[1].set_xlabel("correlation of the limitations")
ax[1].set_ylabel("excluded share")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The shaded band holds every share that the three limitations can produce. The simulation stays inside it and falls as the limitations come together, because the same patients carry several of them.

**Observe and reflect.** Which age group carries most of the exclusion in the simulation? What would a change that removes only one demand, such as the time limit, do for that group?

## 2. The Web Content Accessibility Guidelines (WCAG) 2.2: An audit of the app &#91;[8](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-8)&#93;

*Lecture: [WCAG 2.2: Four principles and testable criteria](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#wcag)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate_patients" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** WCAG 2.2 turns accessibility into testable success criteria. Python reads the audit of the clinic app as a table, counts the failures by principle and level, and estimates what an automated tool alone would find.

**From the formulas to the code.** A page conforms to level AA only if all $m = 55$ criteria of levels A and AA are met. If each criterion failed independently with the probability $q$, a page would conform with the probability $(1 - q)^m$, which `p_conform(q)` computes.

In [ ]:
AUDIT_CSV = """sc;name;level;tools;current;redesign
1.1.1;Non-text Content;A;1;fail;fail
1.2.1;Audio-only and Video-only (Prerecorded);A;0;n/a;n/a
1.2.2;Captions (Prerecorded);A;0;n/a;n/a
1.2.3;Audio Description or Media Alternative (Prerecorded);A;0;n/a;n/a
1.2.4;Captions (Live);AA;0;n/a;n/a
1.2.5;Audio Description (Prerecorded);AA;0;n/a;n/a
1.3.1;Info and Relationships;A;1;fail;pass
1.3.2;Meaningful Sequence;A;0;fail;fail
1.3.3;Sensory Characteristics;A;0;pass;pass
1.3.4;Orientation;AA;0;pass;pass
1.3.5;Identify Input Purpose;AA;1;pass;pass
1.4.1;Use of Color;A;0;fail;pass
1.4.2;Audio Control;A;0;n/a;n/a
1.4.3;Contrast (Minimum);AA;1;fail;pass
1.4.4;Resize Text;AA;0;fail;pass
1.4.5;Images of Text;AA;1;pass;pass
1.4.10;Reflow;AA;0;pass;pass
1.4.11;Non-text Contrast;AA;1;fail;pass
1.4.12;Text Spacing;AA;0;pass;pass
1.4.13;Content on Hover or Focus;AA;0;pass;pass
2.1.1;Keyboard;A;0;fail;pass
2.1.2;No Keyboard Trap;A;0;pass;pass
2.1.4;Character Key Shortcuts;A;0;n/a;n/a
2.2.1;Timing Adjustable;A;0;fail;fail
2.2.2;Pause, Stop, Hide;A;0;pass;pass
2.3.1;Three Flashes or Below Threshold;A;0;pass;pass
2.4.1;Bypass Blocks;A;0;fail;pass
2.4.2;Page Titled;A;1;pass;pass
2.4.3;Focus Order;A;0;fail;pass
2.4.4;Link Purpose (In Context);A;1;pass;pass
2.4.5;Multiple Ways;AA;0;n/a;n/a
2.4.6;Headings and Labels;AA;1;fail;pass
2.4.7;Focus Visible;AA;0;fail;fail
2.4.11;Focus Not Obscured (Minimum);AA;0;pass;pass
2.5.1;Pointer Gestures;A;0;pass;pass
2.5.2;Pointer Cancellation;A;0;pass;pass
2.5.3;Label in Name;A;1;pass;pass
2.5.4;Motion Actuation;A;0;n/a;n/a
2.5.7;Dragging Movements;AA;0;fail;pass
2.5.8;Target Size (Minimum);AA;0;pass;pass
3.1.1;Language of Page;A;1;pass;pass
3.1.2;Language of Parts;AA;1;n/a;n/a
3.2.1;On Focus;A;0;pass;pass
3.2.2;On Input;A;0;pass;pass
3.2.3;Consistent Navigation;AA;0;pass;pass
3.2.4;Consistent Identification;AA;0;pass;pass
3.2.6;Consistent Help;A;0;pass;pass
3.3.1;Error Identification;A;0;fail;pass
3.3.2;Labels or Instructions;A;1;fail;pass
3.3.3;Error Suggestion;AA;0;pass;pass
3.3.4;Error Prevention (Legal, Financial, Data);AA;0;n/a;n/a
3.3.7;Redundant Entry;A;0;pass;pass
3.3.8;Accessible Authentication (Minimum);AA;0;fail;fail
4.1.2;Name, Role, Value;A;1;fail;fail
4.1.3;Status Messages;AA;0;pass;pass"""
audit = pd.read_csv(io.StringIO(AUDIT_CSV), sep=";", dtype={"sc": str})
PRINCIPLES = {"1": "Perceivable", "2": "Operable", "3": "Understandable",
              "4": "Robust"}
audit["principle"] = audit["sc"].str[0].map(PRINCIPLES)
order = list(PRINCIPLES.values())
print(audit.groupby(["principle", "level"]).size().unstack().loc[order])
for design in ("current", "redesign"):
    fails = audit[audit[design] == "fail"]
    by_p = fails["principle"].value_counts().reindex(order, fill_value=0)
    print(f"{design}: {len(fails)} failures |",
          ", ".join(f"{k} {v}" for k, v in by_p.items()))

**Python note.** Each row is one success criterion with its level, whether automated tools can find its failures, and the result of each design. `dtype={"sc": str}` keeps numbers such as 1.4.10 as text, and `.str[0]` takes their first character, the principle.

In [ ]:
def p_conform(q, m=55):
    """Probability that m criteria all pass if each fails with probability q."""
    return (1 - q) ** m

print("q = 0.02:", round(p_conform(0.02), 3))
for design in ("current", "redesign"):
    fails = audit[audit[design] == "fail"]
    found = int(fails["tools"].sum())
    print(f"{design}: tools can find {found} of {len(fails)} failures "
          f"({found / len(fails):.0%}); people must find the others")
    print("   missed by tools:",
          ", ".join(fails.loc[fails["tools"] == 0, "sc"]))

**Python note.** `fails["tools"].sum()` adds the ones in the column, so it counts the failures that a tool could report. The format `:.0%` prints a share as a percentage without decimals.

**Exercise B.** Suppose each of the 55 criteria fails with the probability 0.01. Store the probability that a page conforms in `p_ok`.

In [ ]:
p_ok = None   # your answer

check("Exercise B", p_ok)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_ok = p_conform(0.01)

check("Exercise B", p_ok)

In [ ]:
# Figure: the criteria by principle and level, and the fragility of conformance
counts = audit.groupby(["principle", "level"]).size().unstack().loc[order]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
counts.plot(kind="bar", stacked=True, ax=ax[0], color=["#5A4BB5", "#C9C3E6"],
            rot=0)
ax[0].set_ylabel("success criteria")
qs = np.linspace(0, 0.1, 101)
ax[1].plot(qs, p_conform(qs), color="#5A4BB5")
ax[1].plot(0.02, p_conform(0.02), "o", color="#B53F2E")
ax[1].set_xlabel("probability that one criterion fails")
ax[1].set_ylabel("probability of conformance")
plt.tight_layout()
plt.show()

**Python note.** `counts.plot(kind="bar", stacked=True)` draws a bar per principle with the criteria of level A at the bottom and those of level AA on top. The right panel shows how quickly conformance becomes unlikely when each criterion carries a small risk.

**Observe and reflect.** Which failures of the redesign would an automated tool miss? Who would notice them first: An expert, a patient with a screen reader or a patient with a tremor?

## 3. Reach and sight: Your kiosk &#91;[9](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-9)&#93;

*Lecture: [Reach and sight at the kiosk](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#reach-and-sight)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "p_conform" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Whether a patient can use the kiosk at all depends on the height of its buttons and the angle of its screen. Python computes the share of patients who reach all buttons for any height of the screen, finds the best height and tilt, and replays your kiosk of Part A.

**From the formulas to the code.** For a group whose lowest reach is $L \sim N(\mu_L, \sigma_L)$ and highest reach $H \sim N(\mu_H, \sigma_H)$, `reach_share(y_low, y_top, group, knee)` computes $P_g = \Phi((y_{low} - \mu_L)/\sigma_L)\,\Phi((\mu_H - y_{top})/\sigma_H)$. `coverage(centre, knee)` weights the groups, $C = \sum_g w_g P_g$, and `view_angle(eye, centre, d, tilt)` computes $\theta = |\arctan((e - c)/d) - \tau|$.

In [ ]:
REACH_GROUPS = {   # share; lowest and highest reach (mean, sd) in m; eyes
    "standing, under 65": dict(
        w=0.62, L=(0.6, 0.08), H=(1.78, 0.09), Hk=(1.78, 0.09),
        eye=1.6, d=0.45, dk=0.45),
    "standing, 65 and over": dict(
        w=0.33, L=(0.8, 0.09), H=(1.62, 0.1), Hk=(1.62, 0.1),
        eye=1.55, d=0.45, dk=0.45),
    "wheelchair": dict(
        w=0.05, L=(0.68, 0.08), H=(1.05, 0.08), Hk=(1.34, 0.07),
        eye=1.2, d=0.7, dk=0.45)}
BAND = 0.13      # the rows of buttons lie 0.13 m above and below the centre

def reach_share(y_low, y_top, g, knee=False):
    """Share of group g whose reach includes both rows of buttons."""
    p = REACH_GROUPS[g]
    mh, sh = p["Hk"] if knee else p["H"]
    ml, sl = p["L"]
    return (stats.norm.cdf((y_low - ml) / sl)
            * stats.norm.cdf((mh - y_top) / sh))

def coverage(centre, knee=False):
    """Share of all patients who reach the top and the bottom row."""
    return sum(p["w"] * reach_share(centre - BAND, centre + BAND, g, knee)
               for g, p in REACH_GROUPS.items())

def view_angle(eye, centre, d, tilt):
    """Angle in degrees between the sight line and the screen normal."""
    return abs(math.degrees(math.atan((eye - centre) / d)) - tilt)

for g in REACH_GROUPS:
    print(f"{g}: {reach_share(1.29, 1.55, g):.3f} reach the current rows")
print("current kiosk, centre 1.42 m:", round(coverage(1.42), 3))
print("centre 1.10 m, knee space:", round(coverage(1.10, True), 3))
print("tall man, eyes 1.80 m:", round(view_angle(1.80, 1.42, 0.45, 0), 1),
      "degrees")

**Python note.** `stats.norm.cdf(z)` is the share of a normal distribution below $z$ standard deviations. The first factor is the share whose lowest reach lies below the bottom row, the second the share whose highest reach lies above the top row.

**Exercise C.** What share of the standing patients aged 65 and over reach a top row at 1.55 m? Their highest reach has a mean of 1.62 m and a standard deviation of 0.10 m. Use `stats.norm.cdf` and store the share in `p_old`.

In [ ]:
p_old = None   # your answer

check("Exercise C", p_old)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_old = stats.norm.cdf((1.62 - 1.55) / 0.10)

check("Exercise C", p_old)

In [ ]:
# The best height of the screen, with and without knee space
centres = np.round(np.arange(0.80, 1.70, 0.01), 2)
best = {}
for knee in (False, True):
    cov = [coverage(c, knee) for c in centres]
    k = int(np.argmax(cov))
    best[knee] = (centres[k], cov[k])
    print(f"knee space {knee}: best centre {centres[k]:.2f} m, "
          f"coverage {cov[k]:.3f}")
c_best = best[True][0]
eyes = [(p["eye"], p["dk"]) for p in REACH_GROUPS.values()]
slopes = [math.degrees(math.atan((e - c_best) / d)) for e, d in eyes]
tilt = (max(slopes) + min(slopes)) / 2
print(f"best tilt {tilt:.1f} degrees, worst angle {max(slopes) - tilt:.1f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
for g in REACH_GROUPS:
    ax.plot(centres, [reach_share(c - BAND, c + BAND, g, True)
                      for c in centres], label=g)
ax.plot(centres, [coverage(c, True) for c in centres], color="black",
        lw=2.5, label="all patients")
ax.axvline(1.42, ls=":", color="#B53F2E")
ax.set_xlabel("centre of the screen (m), with knee space")
ax.set_ylabel("share who reach both rows")
ax.legend(fontsize=8)
plt.show()

**Python note.** The best tilt lies halfway between the steepest and the flattest sight line, so that the worst angle is as small as possible. The dotted line marks the kiosk as installed.

**Paste your results.** Copy the results of Part A of the interactive lab and paste them between the triple quotes of `MY_REACH` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_REACH = """
"""
DEFAULT_REACH = """
round,name,group,centre_m,tilt_deg,knee,low_m,high_m,eye_m,angle_deg,reached,served
1,Ayşe,"standing, under 65",1.42,0,0,0.58,1.8,1.55,16.1,1,1
1,Hasan,"standing, 65 and over",1.42,0,0,0.82,1.58,1.6,21.8,1,1
1,Elif,"wheelchair",1.42,0,0,0.66,1.02,1.18,18.9,0,0
1,Mehmet,"standing, 65 and over",1.42,0,0,0.85,1.66,1.63,25.0,1,1
1,Zeynep,"standing, under 65",1.42,0,0,0.52,1.6,1.4,2.5,1,1
1,Fatma,"standing, 65 and over",1.42,0,0,0.88,1.5,1.45,3.8,0,0
1,Burak,"standing, under 65",1.42,0,0,0.62,2.0,1.8,40.2,1,0
1,Emine,"wheelchair",1.42,0,0,0.72,1.08,1.22,15.9,0,0
1,Can,"standing, under 65",1.42,0,0,0.6,1.78,1.66,28.1,1,1
1,Selin,"standing, under 65",1.42,0,0,0.85,1.76,1.52,12.5,1,1
2,Ayşe,"standing, under 65",1.12,28,1,0.58,1.8,1.55,15.7,1,1
2,Hasan,"standing, 65 and over",1.12,28,1,0.82,1.58,1.6,18.8,1,1
2,Elif,"wheelchair",1.12,28,1,0.66,1.3,1.18,20.4,1,1
2,Mehmet,"standing, 65 and over",1.12,28,1,0.85,1.66,1.63,20.6,1,1
2,Zeynep,"standing, under 65",1.12,28,1,0.52,1.6,1.4,3.9,1,1
2,Fatma,"standing, 65 and over",1.12,28,1,0.88,1.5,1.45,8.3,1,1
2,Burak,"standing, under 65",1.12,28,1,0.62,2.0,1.8,28.5,1,1
2,Emine,"wheelchair",1.12,28,1,0.72,1.36,1.22,15.5,1,1
2,Can,"standing, under 65",1.12,28,1,0.6,1.78,1.66,22.2,1,1
2,Selin,"standing, under 65",1.12,28,1,0.85,1.76,1.52,13.6,1,1"""
src = MY_REACH.strip() or DEFAULT_REACH.strip()
print("Data:", "your own kiosk from the interactive lab" if MY_REACH.strip()
      else "the results of a typical student")
reach = pd.read_csv(io.StringIO(src))
for rnd, part in reach.groupby("round"):
    s = part.iloc[0]
    print(f"round {rnd}: centre {s.centre_m} m, tilt {s.tilt_deg} degrees,"
          f" knee space {bool(s.knee)} | served {part.served.sum()} of "
          f"{len(part)} | all patients: {coverage(s.centre_m, bool(s.knee)):.3f}")
    missed = part[part.served == 0]
    for _, r in missed.iterrows():
        why = "reach" if not r.reached else f"angle {r.angle_deg} degrees"
        print(f"   not served: {r['name']} ({r.group}), {why}")

**Python note.** The ten patients of Part A are a small sample, so `coverage` applies your settings to all patients of the clinic with the model of this section.

**Explore.** Move the sliders: The panel shows the share of each group that reaches both rows and the angle of its sight line. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(centre=1.42, tilt=0, knee=False):
    for g, p in REACH_GROUPS.items():
        d = p["dk"] if knee else p["d"]
        share = reach_share(centre - BAND, centre + BAND, g, knee)
        print(f"{g:>22}: reach {share:.3f}, angle "
              f"{view_angle(p['eye'], centre, d, tilt):.1f} degrees")
    print(f"{'all patients':>22}: {coverage(centre, knee):.3f}")

widgets.interact(explore,
                 centre=widgets.FloatSlider(value=1.42, min=0.9, max=1.6,
                                            step=0.01),
                 tilt=widgets.IntSlider(value=0, min=0, max=40),
                 knee=widgets.Checkbox(value=False));

**Observe and reflect.** Which group decides the best height of the screen, and which group decides the best tilt? Would a second, lower kiosk be better than one kiosk for everyone?

## 4. Plain words: Readability in Python &#91;[13](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-13)&#93;

*Lecture: [Plain words: Reading level and readability](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#plain-words)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "coverage" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A text that patients cannot read excludes them as surely as a high button. Python counts words, sentences and syllables and computes the Flesch Reading Ease and the Flesch-Kincaid grade. For Turkish it computes the formula of Ateşman, and it replays your texts of Part C.

**From the formulas to the code.** With $W$ words, $S$ sentences and $Y$ syllables, `fre(text)` computes $206.835 - 1.015\,(W/S) - 84.6\,(Y/W)$, `fkgl(text)` computes $0.39\,(W/S) + 11.8\,(Y/W) - 15.59$, and `atesman(text)` computes $198.825 - 40.175\,(Y/W) - 2.610\,(W/S)$. The ratios are rounded to three decimals, as in the lecture.

In [ ]:
def syllables_en(word):
    """Approximate English syllables: runs of vowels, minus a silent e."""
    w = re.sub(r"[^a-z]", "", word.lower())
    if not w:
        return 0
    n = len(re.findall(r"[aeiouy]+", w))
    if n > 1 and w.endswith("e") and not w.endswith(("le", "ee")):
        n -= 1
    elif (n > 1 and w.endswith(("ed", "es"))
          and not w.endswith(("ted", "ded", "ses", "zes", "ces", "ges",
                              "xes", "ches", "shes"))):
        n -= 1
    return max(1, n)

def syllables_tr(word):
    """Turkish syllables: every syllable holds exactly one vowel."""
    return len(re.findall(r"[aeıioöuüAEIİOÖUÜâîû]", word))

LETTERS = r"[A-Za-zÇĞİÖŞÜçğıöşüâîû]+"

def text_counts(text, lang="en"):
    """Words, sentences and syllables of a text."""
    words = re.findall(LETTERS, text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    count = syllables_en if lang == "en" else syllables_tr
    return len(words), len(sentences), sum(count(w) for w in words)

def ratios(text, lang="en"):
    """Words per sentence and syllables per word, rounded as in the lecture."""
    w, s, y = text_counts(text, lang)
    return round(w / s, 3), round(y / w, 3)

def fre(text):
    wps, spw = ratios(text)
    return 206.835 - 1.015 * wps - 84.6 * spw

def fkgl(text):
    wps, spw = ratios(text)
    return 0.39 * wps + 11.8 * spw - 15.59

def atesman(text):
    wps, spw = ratios(text, "tr")
    return 198.825 - 40.175 * spw - 2.610 * wps

for key in ("current", "plain"):
    print(f"{key}: words, sentences, syllables {text_counts(TEXTS[key])}"
          f" | FRE {fre(TEXTS[key]):.1f} | FKGL {fkgl(TEXTS[key]):.2f}")
for key in ("current_tr", "plain_tr"):
    print(f"{key}: {text_counts(TEXTS[key], 'tr')} | Ateşman "
          f"{atesman(TEXTS[key]):.1f}")

**Python note.** `w.endswith(("le", "ee"))` accepts a tuple and is true if the word ends in any of its endings. The counter is a rule of thumb: It treats *registered* and *minutes* right, but a word such as *prior* would get one syllable instead of two.

**Exercise D.** Compute the Flesch-Kincaid grade of the plain confirmation, `TEXTS["plain"]`, by hand from its counts and then with `fkgl`, and store it in `grade_plain`.

In [ ]:
grade_plain = None   # your answer

check("Exercise D", grade_plain)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
grade_plain = fkgl(TEXTS["plain"])

check("Exercise D", grade_plain)

In [ ]:
# Figure: the sentences of both texts and the share of patients who can read
def sentence_table(text):
    rows = []
    for s in [x for x in re.split(r"(?<=[.!?])\s+", text) if x.strip()]:
        w, _, y = text_counts(s)
        rows.append((w, y / w))
    return rows

READ_MEAN, READ_SD = 8.0, 2.5   # reading levels of the patients (grades)
def can_read(grade):
    """Share of the clinic's patients whose reading level reaches a grade."""
    return 1 - stats.norm.cdf((grade - READ_MEAN) / READ_SD)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for key, colour in (("current", "#B53F2E"), ("plain", "#2E8B57")):
    rows = sentence_table(TEXTS[key])
    ax[0].scatter([r[0] for r in rows], [r[1] for r in rows], s=60,
                  color=colour, label=key)
    g = fkgl(TEXTS[key])
    ax[1].plot(g, can_read(g), "o", color=colour, ms=9)
    ax[1].annotate(f"{key}: {can_read(g):.3f}", (g, can_read(g)),
                   xytext=(6, 6), textcoords="offset points")
grades = np.linspace(0, 16, 161)
ax[1].plot(grades, can_read(grades), color="#585E6E")
ax[0].set_xlabel("words in the sentence")
ax[0].set_ylabel("syllables per word")
ax[0].legend()
ax[1].set_xlabel("Flesch-Kincaid grade of the text")
ax[1].set_ylabel("share who can read it")
plt.tight_layout()
plt.show()

**Python note.** The pattern `(?<=[.!?])\s+` splits after a full stop, exclamation mark or question mark and keeps the mark with its sentence. `can_read` assumes reading levels that follow a normal distribution with a mean of grade 8 and a standard deviation of 2.5 grades.

**Paste your results.** Copy the results of Part C of the interactive lab and paste them between the triple quotes of `MY_TEXTS` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
READ_SLOTS = [   # the versions a, b and c of each sentence; 0 drops a fact
    [
        [('Your appointment request has been successfully registered in the '
          'outpatient scheduling system.', 1),
         ('Your appointment request has been registered.', 1),
         ('Your appointment is booked.', 1)],
        [('Please present yourself at the registration desk fifteen minutes '
          'before your scheduled appointment time.', 1),
         ('Please arrive at the front desk fifteen minutes before your '
          'appointment.', 1),
         ('Please come early.', 0)],
        [('Presentation of a valid identification document is mandatory for '
          'all outpatient procedures.', 1),
         ('You must bring a valid identity document.', 1),
         ('Bring your health card.', 1)],
        [('Cancellations must be communicated at least twenty-four hours in '
          'advance.', 1),
         ('Please tell us one day before if you have to cancel.', 1),
         ('If you cannot come, cancel one day before.', 1)]],
    [
        [('Authentication failure: the entered identifier does not '
          'correspond to any registered patient record.', 1),
         ('The patient number that you entered could not be found in our '
          'records.', 1),
         ('We could not find this number.', 1)],
        [('Please verify the accuracy of the information provided and '
          'reattempt the operation.', 1),
         ('Please check the number carefully and try to enter it again.', 1),
         ('Try again.', 0)],
        [('The identifier is printed on the reverse side of your hospital '
          'identification card.', 1),
         ('You will find the number printed on the back of your health '
          'insurance card.', 1),
         ('It is on the back of your card.', 1)],
        [('For further assistance, kindly contact the administrative '
          'personnel at the registration desk.', 1),
         ('If the problem continues, please ask the staff at the '
          'registration desk.', 1),
         ('Ask at the front desk.', 1)]]]
MY_TEXTS = """
"""
DEFAULT_TEXTS = """
round,choice,words,sentences,syllables,fre,fkgl,facts,understood
1,cbcb,30,4,40,86.5,3.06,1,10
2,bbcc,37,4,47,90.0,3.00,1,11"""
src = MY_TEXTS.strip() or DEFAULT_TEXTS.strip()
print("Data:", "your own texts from the interactive lab" if MY_TEXTS.strip()
      else "the texts of a typical student")
texts = pd.read_csv(io.StringIO(src))
for _, r in texts.iterrows():
    slots = READ_SLOTS[int(r["round"]) - 1]
    chosen = [slots[i]["abc".index(ch)] for i, ch in enumerate(r["choice"])]
    text = " ".join(c[0] for c in chosen)
    g = fkgl(text)
    print(f"round {r['round']} ({r['choice']}): FKGL {g:.2f} (game "
          f"{r['fkgl']:.2f}), facts kept {all(c[1] for c in chosen)}, "
          f"{r['understood']} of 12 understood, clinic {can_read(g):.3f}")
    print("   ", text)

**Python note.** Each choice such as `cbcb` names the version of the four sentences. The code rebuilds the text, computes its grade again and applies `can_read` to all patients of the clinic.

**Observe and reflect.** Pick one sentence of the current confirmation and rewrite it yourself. How far does the grade fall, and which fact could your version lose?

## 5. Screen readers: The cost of navigation &#91;[19](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-19)&#93;

*Lecture: [Screen readers, focus order and the keyboard](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#screen-readers)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "fkgl" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A screen-reader user hears a screen one element after another. Python counts the key presses and the listening time of the booking of Part D, for the current app and the app of stage 10. It compares them with your own run.

**From the formulas to the code.** For a target that is equally likely at each of $n$ positions, reading element by element needs $(n + 1)/2$ presses on average, which `presses_linear(n)` computes. With $s$ sections behind headings, `presses_headings(n, s)` computes $(s + 1)/2 + (n/s + 1)/2$. `screen_time(items, k)` adds $t_K$ per press and the words heard up to element $k$ at the listening rate $r$.

In [ ]:
KEY_S = 0.28           # a key press of the keystroke-level model (s)
LISTEN_WPM = 334       # listening rate of screen-reader users (words/minute)

def presses_linear(n):
    """Average presses to reach one of n elements, one at a time."""
    return (n + 1) / 2

def presses_headings(n, s):
    """Average presses with s headings: jump to the section, then step."""
    return (s + 1) / 2 + (n / s + 1) / 2

def words(item):
    return len(re.findall(r"[A-Za-z0-9:]+", item.lstrip("# ")))

def screen_time(items, k, wpm=LISTEN_WPM):
    """Seconds to press k times and hear the first k elements of a screen."""
    heard = sum(words(x) for x in items[:k])
    return k * KEY_S + heard / (wpm / 60)

print("24 elements:", presses_linear(24), "presses one by one,",
      presses_headings(24, 4), "with 4 headings")
print("one element of 4 words:", round(screen_time(["a b c d"], 1), 3), "s")

**Python note.** `item.lstrip("# ")` removes the mark of a heading before the words are counted. The listening rate is divided by 60 to turn words per minute into words per second.

**Exercise E.** A screen of 30 elements has 5 headings with 6 elements each. Compute the average number of presses with headings and store it in `presses_h`.

In [ ]:
presses_h = None   # your answer

check("Exercise E", presses_h)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
presses_h = presses_headings(30, 5)

check("Exercise E", presses_h)

In [ ]:
NAV_SCREENS = {     # what a screen reader reads, in order; '# ' marks a heading
    "current": [
        ['image', 'City Hospital', 'button', 'button', 'Book an appointment',
         'Step 1 of 4', 'Dermatology, button', 'Neurology, button',
         'Eye diseases, button', 'Cardiology, button', 'Orthopaedics, button',
         'Ear, nose and throat, button', 'button'],
        ['image', 'City Hospital', 'button', 'button', 'Cardiology', 'Step 2 of 4',
         'Monday, button', 'Tuesday, button', 'Wednesday, button',
         'Thursday, button', 'button'],
        ['image', 'City Hospital', 'button', 'button', 'Wednesday', 'Step 3 of 4',
         '08:30, button', '11:30, button', '09:30, button', '12:30, button',
         '10:30, button', '13:30, button', 'button'],
        ['image', 'City Hospital', 'button', 'button', 'Summary', 'Step 4 of 4',
         'Department', 'Day', 'Time', 'Cardiology', 'Wednesday', '10:30', 'button',
         'button']],
    "stage 10": [
        ['# Book an appointment. Step 1 of 4: Choose a department',
         'Cardiology, button', 'Dermatology, button', 'Eye diseases, button',
         'Neurology, button', 'Orthopaedics, button', 'Ear, nose and throat, button',
         '# Other actions', 'Back, button', 'Help, button',
         'Language: English, button'],
        ['# Step 2 of 4: Choose a day for cardiology', 'Monday, button',
         'Tuesday, button', 'Wednesday, button', 'Thursday, button',
         '# Other actions', 'Back, button', 'Help, button'],
        ['# Step 3 of 4: Choose a time on Wednesday', '08:30, button',
         '09:30, button', '10:30, button', '11:30, button', '12:30, button',
         '13:30, button', '# Other actions', 'Back, button', 'Help, button'],
        ['# Step 4 of 4: Check and confirm', 'Department: Cardiology',
         'Day: Wednesday', 'Time: 10:30', 'Confirm the booking, button',
         'Change, button', '# Other actions', 'Back, button', 'Help, button']]}
NAV_TARGETS = {"current": [10, 9, 11, 13], "stage 10": [2, 4, 4, 5]}

for version, screens in NAV_SCREENS.items():
    total_p, total_t = 0, 0.0
    for items, k in zip(screens, NAV_TARGETS[version]):
        total_p += k
        total_t += screen_time(items, k)
    print(f"{version}: at least {total_p} presses, about {total_t:.1f} s "
          f"for the booking")

**Python note.** Each screen starts with the cursor on its first element. Reaching the element at position $k$ takes $k - 1$ presses of the arrow and one press to activate it, so $k$ presses in all.

**Paste your results.** Copy the results of Part D of the interactive lab and paste them between the triple quotes of `MY_NAV` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_NAV = """
"""
DEFAULT_NAV = """
round,version,screen,presses,wrong,time_s
1,current,1,13,1,31.5
1,current,2,9,0,18.2
1,current,3,14,0,29.0
1,current,4,13,0,27.4
2,stage 10,1,2,0,6.1
2,stage 10,2,4,0,9.8
2,stage 10,3,5,0,10.4
2,stage 10,4,5,0,11.2"""
src = MY_NAV.strip() or DEFAULT_NAV.strip()
print("Data:", "your own run from the interactive lab" if MY_NAV.strip()
      else "the run of a typical student")
nav = pd.read_csv(io.StringIO(src))
nav["least"] = [NAV_TARGETS[v][s - 1] for v, s in zip(nav.version, nav.screen)]
summary = nav.groupby("version", sort=False)[
    ["presses", "least", "wrong", "time_s"]].sum()
summary["extra"] = summary["presses"] - summary["least"]
print(summary)

# Figure: presses against the number of elements, and time against listening
n = np.arange(4, 61)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(n, presses_linear(n), label="one by one")
ax[0].plot(n, presses_headings(n, 4), label="4 headings")
ax[0].plot(n, presses_headings(n, np.sqrt(n)), label="sqrt(n) headings")
ax[0].set_xlabel("elements on the screen")
ax[0].set_ylabel("average presses")
ax[0].legend(fontsize=8)
rates = np.arange(120, 461, 20)
for version, screens in NAV_SCREENS.items():
    t = [sum(screen_time(i, k, r) for i, k in zip(screens,
             NAV_TARGETS[version])) for r in rates]
    ax[1].plot(rates, t, label=version)
ax[1].set_xlabel("listening rate (words per minute)")
ax[1].set_ylabel("time for the booking (s)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** With $\sqrt{n}$ sections of $\sqrt{n}$ elements each, the two parts of the formula are equal, which makes the sum smallest for a given $n$.

**Explore.** Move the sliders: The panel compares the average presses one by one and with headings. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(n=24, s=4):
    s = min(s, n)
    print(f"one by one: {presses_linear(n):.1f} presses | with {s} headings:"
          f" {presses_headings(n, s):.1f} presses")

widgets.interact(explore, n=widgets.IntSlider(value=24, min=2, max=80),
                 s=widgets.IntSlider(value=4, min=1, max=20));

**Observe and reflect.** On which screen of Part D did you lose most presses, and which element made you go back? How would you change the order of that screen?

## 6. The clinic app, stage 10 &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#ref-6)&#93;

*Lecture: [The clinic app, stage 10: An accessibility audit of the app and the kiosk](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#the-clinic-app-stage-10)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "NAV_SCREENS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 10 audits the clinic app and its kiosk. Python collects the results of the sections in one dictionary. It holds the failures of the audit, the share of patients who reach the kiosk and the reading level of the confirmation. It adds the presses of a screen-reader user and the time limit.

In [ ]:
STAGE10 = {}
for design in ("current", "redesign"):
    fails = audit[audit[design] == "fail"]
    STAGE10[f"WCAG failures, {design}"] = len(fails)
    STAGE10[f"found by tools, {design}"] = int(fails["tools"].sum())
STAGE10["patients who reach the kiosk, as installed"] = round(coverage(1.42), 3)
STAGE10["patients who reach the kiosk, stage 10"] = round(coverage(1.13, True), 3)
STAGE10["tilt of the screen, stage 10 (degrees)"] = round(tilt, 1)
for key in ("current", "plain"):
    g = fkgl(TEXTS[key])
    STAGE10[f"grade of the confirmation, {key}"] = round(g, 2)
    STAGE10[f"patients who can read it, {key}"] = round(can_read(g), 3)
STAGE10["Ateşman score, Turkish, current"] = round(atesman(TEXTS["current_tr"]), 1)
STAGE10["Ateşman score, Turkish, plain"] = round(atesman(TEXTS["plain_tr"]), 1)
for version, screens in NAV_SCREENS.items():
    STAGE10[f"screen-reader presses, {version}"] = sum(NAV_TARGETS[version])
    STAGE10[f"screen-reader time, {version} (s)"] = round(sum(
        screen_time(i, k) for i, k in zip(screens, NAV_TARGETS[version])), 1)

# The time limit: 30 s per screen, log-normal times of patients aged 65 and
# over with a median of 12 s and a spread of 0.55 (Week 9)
p_screen = round(1 - stats.norm.cdf((math.log(30) - math.log(12)) / 0.55), 3)
STAGE10["time-out on one screen, 65 and over"] = round(p_screen, 3)
STAGE10["time-out in four screens, 65 and over"] = round(
    1 - (1 - p_screen) ** 4, 3)
for key, value in STAGE10.items():
    print(f"{key}: {value}")

**Python note.** The time limit uses the log-normal times of Week 9: The logarithm of 30 s lies $(\ln 30 - \ln 12)/0.55$ standard deviations above the logarithm of the median. The share for one screen is rounded to three decimals before the four screens are combined, as in the lecture.

**Exercise F.** Store the share of all patients who reach both rows of the stage 10 kiosk, with its centre at 1.13 m and knee space, in `stage_cover`.

In [ ]:
stage_cover = None   # your answer

check("Exercise F", stage_cover)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
stage_cover = coverage(1.13, True)

check("Exercise F", stage_cover)

In [ ]:
# Figure: the audit, the kiosk and the confirmation of stage 10
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
f = [STAGE10["WCAG failures, current"], STAGE10["WCAG failures, redesign"], 0]
ax[0].bar(["current", "redesign", "stage 10"], f,
          color=["#B53F2E", "#C98510", "#2E8B57"])
ax[0].set_ylabel("failed criteria (A and AA)")
r = [STAGE10["patients who reach the kiosk, as installed"],
     STAGE10["patients who reach the kiosk, stage 10"]]
ax[1].bar(["as installed", "stage 10"], r, color=["#B53F2E", "#2E8B57"])
ax[1].set_ylim(0, 1)
ax[1].set_ylabel("patients who reach the kiosk")
g = [STAGE10["grade of the confirmation, current"],
     STAGE10["grade of the confirmation, plain"]]
ax[2].bar(["current", "plain"], g, color=["#B53F2E", "#2E8B57"])
ax[2].set_ylabel("Flesch-Kincaid grade")
plt.tight_layout()
plt.show()

**Python note.** Stage 10 counts zero failures after its fixes. An audit by experts is not the end, however: Week 9 showed how to test with patients, and the next test includes patients with disabilities.

## 7. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE10" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-10/"
               "NB10_accessibility.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The methods of this week apply to any app. Each app below has a message that users read, the elements of one screen, its headings and its time limit with typical times per screen.

The values are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# clinic, banking, e_government, food_delivery, course_registration
APPS = {
    "clinic": (TEXTS["current"], 13, 1, 30, 12, 4),
    "banking": ("Your transaction could not be completed because the "
                "authentication session has expired. Please reinitiate the "
                "transfer procedure.", 38, 5, 120, 25, 5),
    "e_government": ("Applicants are required to upload a certified copy of "
                     "the relevant documentation prior to the submission "
                     "deadline.", 52, 7, 900, 90, 6),
    "food_delivery": ("Your order is on its way. The courier will call you "
                      "when he is near.", 24, 4, 300, 20, 3),
    "course_registration": ("Registration for elective courses is "
                            "conditional upon the fulfilment of the "
                            "prerequisite requirements.", 44, 6, 600, 60, 4)}

message, n, s, limit, median, screens = APPS[APP]
g = fkgl(message)
print(f"{APP}: grade {g:.2f}, reading ease {fre(message):.1f}; "
      f"{can_read(g):.3f} of readers like the patients can read it")
print(f"  {n} elements: {presses_linear(n):.1f} presses one by one, "
      f"{presses_headings(n, s):.1f} with {s} headings")
p = 1 - stats.norm.cdf((math.log(limit) - math.log(2 * median)) / 0.55)
print(f"  time limit {limit} s: a slow user (median {2 * median} s) times "
      f"out on {p:.3f} of the screens, {1 - (1 - p) ** screens:.3f} at "
      f"least once in {screens} screens")

**Python note.** Each entry holds a message of the app, the elements of one screen and its headings, and the time limit of a screen. It also holds the median time per screen and the number of screens. A slow user takes twice the median time.

**Observe and reflect.** Which barrier of your app excludes the most users: The message, the navigation or the time limit? Which one would you fix first?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a function that flags every sentence of a text with more than 20 words or a Flesch-Kincaid grade above 8, and run it on the error messages of a program. |
| Web development | Count the headings and the focusable elements of a web page that you use, and compute the average presses one by one and with headings. |
| Mobile development | Measure the targets of three buttons of a mobile app in CSS pixels and in millimetres, and check them against the minimum of WCAG 2.2. |
| Game development | Rewrite the tutorial text of a game in plain words and compare the grades of both versions with `fkgl`. |
| Embedded systems and the Internet of Things | Choose the mounting height of a wall panel, such as a thermostat, with `coverage`, and give the share it serves. |
| Data science | Simulate the exclusion of a design with four demands and plot how the excluded share changes with the correlation of the limitations. |
| Cybersecurity | Compare two ways of signing in, a password and a code sent to the phone, against the criterion on accessible authentication, and estimate the time each one needs. |
| Health informatics | Compute the Ateşman score of a Turkish patient leaflet and rewrite its hardest sentence. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-10).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>